# MNIST MLP3 — SGD + Nesterov vs Full Matrix-Log RG
Matches the baseline optimizer recipe. WeightWatcher refreshes the midpoint support once per epoch; the full matrix-log projector is applied to completed SGD steps. A bounded validation-only grid search selects projection strength, correction cap, and cadence before the final three-seed run.


In [ ]:
from pathlib import Path
import copy, itertools, os, sys
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from IPython.display import display
REPO=None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p/'baseline'/'rg_baselines').is_dir() and (p/'optimizers').is_dir(): REPO=p; break
if REPO is None: raise RuntimeError('Run from a clone of CalculatedContent/rg_optimizers')
for p in [REPO/'baseline', REPO/'optimizers'/'full_matrix_log_rg']:
    if str(p) not in sys.path: sys.path.insert(0,str(p))
from rg_baselines import BaselineConfig, DEFAULT_BASELINE_SEEDS, MLP3, set_scheduled_learning_rates
from rg_baselines.engine import choose_device, evaluate, set_seed
from full_matrix_log_rg import FullMatrixLogConfig, FullMatrixLogRG, analyze_supports
DEVICE=choose_device(); DATA_DIR=Path(os.environ.get('RG_BASELINE_DATA_DIR',REPO/'baseline'/'data')); DATA_DIR.mkdir(parents=True,exist_ok=True)
BASE=BaselineConfig(optimizer='sgd_momentum',epochs=30,validation_size=5000,sgd_learning_rate=0.05,sgd_min_learning_rate=5e-4,sgd_warmup_epochs=2,sgd_momentum=0.90,sgd_nesterov=True,sgd_weight_decay=1e-4)
SEEDS=DEFAULT_BASELINE_SEEDS


In [ ]:
def loaders(seed):
    tr=transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.1307,),(0.3081,))])
    full=datasets.MNIST(str(DATA_DIR),train=True,download=True,transform=tr); test=datasets.MNIST(str(DATA_DIR),train=False,download=True,transform=tr)
    g=torch.Generator().manual_seed(int(BASE.split_seed)); perm=torch.randperm(len(full),generator=g).tolist(); vi,ti=perm[:5000],perm[5000:]
    tg=torch.Generator().manual_seed(int(seed)+101); workers=0 if DEVICE.type=='mps' else int(BASE.num_workers); common=dict(num_workers=workers,pin_memory=DEVICE.type=='cuda')
    return (DataLoader(Subset(full,ti),batch_size=BASE.batch_size,shuffle=True,generator=tg,**common),DataLoader(Subset(full,ti),batch_size=BASE.batch_size,shuffle=False,**common),DataLoader(Subset(full,vi),batch_size=BASE.batch_size,shuffle=False,**common),DataLoader(test,batch_size=BASE.batch_size,shuffle=False,**common))
def base_sgd(model):
    matrix=[p for p in model.parameters() if p.ndim==2]; other=[p for p in model.parameters() if p.ndim!=2]
    return torch.optim.SGD([{'params':matrix,'weight_decay':1e-4},{'params':other,'weight_decay':0.0}],lr=0.05,momentum=0.90,nesterov=True)
def run_one(seed,rg_cfg=None,epochs=30,fast=False,label='run'):
    set_seed(int(seed)); train,train_eval,val,test=loaders(seed); model=MLP3().to(DEVICE); base=base_sgd(model); opt=base if rg_cfg is None else FullMatrixLogRG(base,model.named_parameters(),rg_cfg)
    steps=len(train); total=epochs*steps; step=0; perf=[]; spec=[]; corr=[]
    if rg_cfg is not None:
        c=analyze_supports(model,epoch=0); opt.set_supports(c.supports); spec.append(c.metrics.assign(seed=seed,label=label))
    for epoch in range(1,epochs+1):
        model.train()
        for x,y in train:
            set_scheduled_learning_rates(opt,BASE,update_index=step,total_steps=total,steps_per_epoch=steps); x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad(set_to_none=True); loss=F.cross_entropy(model(x),y); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); step+=1
            if rg_cfg is not None:
                for n,r in opt.last_stats.items(): corr.append({'seed':seed,'label':label,'epoch':epoch,'parameter_name':n,**r})
        va=evaluate(model,val,device=DEVICE,max_batches=20 if fast else None); te=evaluate(model,test,device=DEVICE,max_batches=20 if fast else None); perf.append({'seed':seed,'label':label,'epoch':epoch,'validation_loss':va['loss'],'validation_accuracy':va['accuracy'],'test_loss':te['loss'],'test_accuracy':te['accuracy']})
        c=analyze_supports(model,epoch=epoch); spec.append(c.metrics.assign(seed=seed,label=label));
        if rg_cfg is not None: opt.set_supports(c.supports)
    return pd.DataFrame(perf),pd.concat(spec,ignore_index=True),pd.DataFrame(corr)


## Validation-only hyperparameter grid
Primary objective: final validation accuracy. Mean |alpha-2| and mean correction ratio are diagnostics/tie-breakers; test metrics do not select the point.


In [ ]:
rows=[]
for strength,cap,every in itertools.product([0.25,0.5,1.0],[0.05,0.10,0.25],[1,5,25]):
    cfg=FullMatrixLogConfig(projection_strength=strength,max_correction_ratio=cap,apply_every_steps=every)
    p,s,c=run_one(SEEDS[0],cfg,epochs=5,fast=True,label='grid')
    rows.append({'projection_strength':strength,'max_correction_ratio':cap,'apply_every_steps':every,'validation_accuracy':p.iloc[-1].validation_accuracy,'validation_loss':p.iloc[-1].validation_loss,'mean_abs_alpha_minus_2':float(np.nanmean(np.abs(s.alpha-2.0))),'mean_correction_ratio':float(c.correction_ratio.mean()) if not c.empty else 0.0})
GRID=pd.DataFrame(rows).sort_values(['validation_accuracy','mean_abs_alpha_minus_2','mean_correction_ratio'],ascending=[False,True,True]).reset_index(drop=True); display(GRID)
b=GRID.iloc[0]; BEST=FullMatrixLogConfig(projection_strength=float(b.projection_strength),max_correction_ratio=float(b.max_correction_ratio),apply_every_steps=int(b.apply_every_steps)); print(BEST)


In [ ]:
PERF=[]; SPEC=[]; CORR=[]
for seed in SEEDS:
    p,s,_=run_one(seed,None,epochs=30,label='SGD + momentum'); PERF.append(p); SPEC.append(s)
    p,s,c=run_one(seed,BEST,epochs=30,label='SGD + momentum + FullMatrixLogRG'); PERF.append(p); SPEC.append(s); CORR.append(c)
PERF=pd.concat(PERF,ignore_index=True); SPEC=pd.concat(SPEC,ignore_index=True); CORR=pd.concat(CORR,ignore_index=True)
display(PERF.groupby(['label','epoch'])[['validation_accuracy','test_accuracy','validation_loss','test_loss']].mean().tail(10))
display(SPEC.sort_values(['label','parameter_name','epoch']))
display(CORR.groupby('parameter_name').agg(mean_phi=('phi','mean'),mean_base_drift=('base_drift','mean'),mean_corrected_drift=('corrected_drift','mean'),mean_correction_ratio=('correction_ratio','mean'),applied_fraction=('applied','mean')))
